# 42장 · 회귀 검사 · 최종 검증 질문으로 후보를 비교하고 실패 사례 남기기

회귀 검사는 이전에 고친 문제가 다음 변경에서 다시 생기는지 확인하는 검사입니다. 입력과 기대 결과를 함께 보관해 두고, 구현을 바꾼 뒤 같은 입력으로 다시 실행합니다.

## 고친 동작을 작은 검사로 남기기

`assert` 뒤의 식이 참이면 다음 줄을 실행합니다. 거짓이면 `AssertionError`에서 멈춥니다. 먼저 앞뒤 공백을 지운 결과를 기대 문자열과 비교해 이전 동작이 유지되는지 확인하세요.

In [ ]:
def clean(text):
    return text.strip()

assert clean("  사과  ") == "사과"
print("통과")

통과


공백 처리를 지우면 같은 입력에서 AssertionError가 납니다. 챗봇에서도 실패했던 질문을 저장해 두고, 다시 답할 때 출처와 행동이 기대값에 맞는지 비교합니다.

## 응용 실습 · 최종 질문으로 후보 비교하기

### 확인 질문을 열기 전에 후보 고정하기

41장에서 저장한 검색 개수와 개발용 질문을 읽습니다. 최종 확인 세트를 본 뒤 값을 바꾸면 그 세트가 개발 자료가 되므로 최초 선택을 파일로 남깁니다. 통합 ZIP을 모두 풀고 실행하세요. 내 파일이 없으면 출력에 표시된 교재 예시로 시작합니다.

In [ ]:
from pathlib import Path
import json
import sys

root = next(folder for folder in (Path.cwd(), *Path.cwd().parents)
            if (folder / "pre-mid/documents/workflow").is_dir())
sys.path.insert(0, str(root / "pre-mid/documents/workflow"))
from workbook_files import read_previous, save_record

previous = read_previous(root, 41)
print("후보 k", previous["candidate_k"])
print("개발에서 본 질문", previous["case"]["inputs"]["question"])

내 작업 · chapter41.json
후보 k 2
개발에서 본 질문 2027년 4월 15일 기준으로 A형의 파일이 나뉘는 간격은 얼마이고 그때 녹음도 멈추나요?


**준비물**. 전체 실습 ZIP을 풀고 Jupyter 커널을 그 환경으로 선택합니다. 앞 장에서 쓰던 패키지에 LangSmith SDK가 필요합니다. 설치가 필요한 커널에서는 별도 셀에 `%pip install "langsmith>=0.4,<1" langchain-openai langgraph chromadb pypdf python-dotenv streamlit`를 실행한 뒤 커널을 재시작하세요. 압축을 푼 폴더의 `.env`에 개인 실습용 `LANGSMITH_API_KEY`, `OPENAI_API_KEY`, 세 장에서 사용할 같은 `OPENAI_MODEL`을 둡니다. LangSmith 워크스페이스가 여럿이면 해당 워크스페이스의 키 또는 `LANGSMITH_WORKSPACE_ID`도 확인하세요. 팀이 선택한 LangSmith 엔드포인트가 있다면 `LANGSMITH_ENDPOINT`도 맞춥니다. 모델 호출은 계정 사용량에 반영됩니다. 원격 기록에는 제공된 가상 제품 자료를 사용합니다.

In [ ]:
from pathlib import Path
import json
import os
import sys
from dotenv import load_dotenv

# 전체 실습 묶음을 푼 폴더 또는 그 안의 장 폴더에서 노트북을 엽니다.
base = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pre-mid/documents/evaluation/eval_lab.py").is_file())
docs = base / "pre-mid/documents"
sys.path.insert(0, str(docs / "evaluation"))
load_dotenv(base / ".env")
from eval_lab import examples

# 키의 실제 값은 출력하지 않습니다. 개인 실습 공간의 키를 사용하세요.
smith_ready = bool(os.getenv("LANGSMITH_API_KEY") or os.getenv("LANGCHAIN_API_KEY"))
model_ready = bool(os.getenv("OPENAI_API_KEY"))
if smith_ready:
    os.environ["LANGSMITH_TRACING"] = "true"
    from langsmith import Client
    client = Client()
work = base / "my-recorder-evaluation"
work.mkdir(exist_ok=True)
print("LangSmith 연결 설정", smith_ready, "/ 모델 연결 설정", model_ready)

## 1. 답을 보기 전에 선택과 기준을 확정하기

**최종 확인용 세트(holdout)** 는 설정을 고르는 동안 따로 보관한 질문입니다. 검토 전 후보와 기준을 적으세요. 예를 들어 “근거가 없는 기능을 새로 지원한다고 답하면 후보를 보류한다. 그 외의 경우 기종·판본·시점·사용 행동이 유지되는지 먼저 보고, 품질이 같으면 지연과 토큰을 본다”는 기준을 사용할 수 있습니다. 채택 기준은 팀이 서비스를 어디에 쓸지에 따라 달라집니다.

아래 셀은 41장의 기준선 k=4와 수업 후보 k=2를 기록합니다. 팀의 개발 실험에서 다른 후보를 선택했다면 최종 질문을 열기 전에 값을 바꾸세요. 팀원이 같은 실험을 재현할 수 있게 41장의 보고서가 있으면 개발 실험 ID도 보관합니다.

검증용으로 떼어 둔 질문을 보기 전에 `candidate_k`와 채택 기준부터 기록합니다. `Path.exists()`는 기록 파일이 있는지 확인하고 `write_text`는 문자열을 파일에 씁니다. 처음 저장할 때만 쓰고 `json.loads`로 다시 읽어 현재 계획과 비교하므로 재실행 중에 기준이 달라지면 그 자리에서 알 수 있습니다.

In [2]:
from datetime import datetime, timezone
candidate_k = previous["candidate_k"]
if not 1 <= candidate_k <= 6 or candidate_k == 4:
    raise ValueError("설명서 여섯 쪽 안에서 기준선 4와 다른 후보를 선택하세요.")
dev_path = work / "dev-results.json"
development = json.loads(dev_path.read_text(encoding="utf-8")) if dev_path.exists() else {}
plan = {"candidate_k": candidate_k, "baseline_k": 4,
        "development_experiments": {key: value["experiment_id"] for key, value in development.items()},
        "rule": "새로운 근거 없는 기능 단정이 생기면 보류. 답할 수 있는 질문은 판본·시점·행동을 대조한다."}
# 평가 기준을 나중에 바꾸었는지 확인할 수 있도록 최초 기록을 보존합니다.
plan_path = work / "decision-plan.json"
if not plan_path.exists():
    plan_path.write_text(json.dumps(plan, ensure_ascii=False, indent=2), encoding="utf-8")
saved_plan = json.loads(plan_path.read_text(encoding="utf-8"))
if saved_plan != plan:
    raise ValueError("기존 계획과 값이 다릅니다. 기존 계획을 확인하고 새 실험은 다른 작업 폴더에 기록하세요.")
print(saved_plan)

{'candidate_k': 2, 'baseline_k': 4, 'development_experiments': {}, 'rule': '새로운 근거 없는 기능 단정이 생기면 보류. 답할 수 있는 질문은 판본·시점·행동을 대조한다.'}


이 셀로 만든 계획은 로컬 기록입니다. 여러 사람이 검토하는 수업에서는 6~7장에서 배운 Git으로 질문을 열기 전의 커밋을 남겨도 좋습니다. 최종 확인을 보고 재차 조정했다면 이번 질문은 이미 개발에 사용한 셈입니다. 동료에게 처음 보는 입력을 다시 받아 마지막 확인을 합니다.

## compare로 같은 최종 질문을 두 구현에 보내기

`compare(client, "holdout", rows, selected, index)`는 선택한 두 구현을 같은 평가 세트에서 실행합니다. `selected`는 비교할 구현이고 `index`는 공통 검색 색인입니다. 실행 오류가 난 질문은 오답이나 보류로 바꿔 적지 말고 오류로 남겨야 두 결과를 해석할 수 있습니다.

`questions/recorder-holdout.csv`는 새 질문 네 건입니다. 아직 답안표를 열지 않고 기종, 적용 시점, 사용자가 하려는 행동에 표시하세요. 기대 답은 평가자에게만 전달합니다. 기준선과 후보가 받는 것은 같은 질문과 사용자가 화면에서 고른 기종·판본입니다.

In [3]:
from eval_lab import read_rows
for item in read_rows("questions", "holdout"):
    print(item["id"], item["question"])

H01 2027년 5월에 B형으로 이미 녹음을 시작했어요. 지금 표시등을 감추려면 어떻게 하나요?
H02 2027년 5월에 A형 화면을 어둡게 하면 녹음 중인 빨간 표시등도 꺼지나요?
H03 2027년 5월 기준으로 B형이 저장 중에 꽉 차면 기존 파일도 없어지나요?
H04 2027년 5월에 A형을 쓰면 녹음할 때 주변 소음을 자동으로 줄여 주나요?


확인 세트를 만들 때 파일을 프로그램으로 읽어도 정답 정보가 챗봇 입력에 들어가지는 않습니다. Datasets 화면에서 Inputs에 `source`, `page`, `reference_answer`가 없는지 따로 확인하세요. k=4가 기준, k=2가 후보라면 아래 셀은 4문항에 두 번씩, 총 여덟 번의 챗봇 실행을 기록합니다.

In [ ]:
from eval_lab import make_targets, compare
live = smith_ready and model_ready
if live:
    from rag_lab import open_index
    index = open_index()
    targets = make_targets(index, candidate_k)
    selected = {name: target for name, target in targets.items() if name.startswith("graph-")}
    holdout = compare(client, "holdout", examples("holdout"), selected, index)
    # 인간 검토 전에 당시의 모델 답·인용·실행 ID를 함께 저장합니다.
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    report_file = work / f"holdout-{stamp}.json"
    report_file.write_text(json.dumps(holdout, ensure_ascii=False, indent=2), encoding="utf-8")
    print("동료가 그대로 열 수 있는 결과", report_file.name)
    for label, report in holdout.items():
        print(label, report["experiment_id"], "완료", len(report["rows"]), "오류", sum(bool(r["error"]) for r in report["rows"]))

키가 없는 독자는 원문과 아래 두 상담 답을 대조하는 판단 연습을 할 수 있습니다. 팀이 실제로 채택 결정을 내릴 때에는 자신의 실험 ID와 원격 실행을 남깁니다. 미실행 행·접속 오류는 보류 답변처럼 채점하지 않습니다.

## 3. 평균이 같은데도 결정이 달라지는 경우

H01을 읽습니다. 상담 답 예시 ① “설정에서 표시등 숨김을 선택하세요.” 예시 ② “이미 녹음 중이면 지금은 설정을 바꿀 수 없습니다. 녹음을 끝낸 뒤 숨김으로 설정하고 새 녹음을 시작하세요.” 두 답 모두 B형 1.0판 1쪽을 인용할 수 있습니다. 사용자가 **지금 할 수 있는 일**을 안내한 쪽을 고르고 원문의 대응 구절을 찾으세요.

H04의 근거 없음과 H03의 저장 공간을 구별하세요. 설명서에 소개되지 않은 기능은 보류 대상으로 삼습니다. 저장 공간이 꽉 찬 경우는 원문 2쪽에 녹음 중단과 기존 파일 보존이 적혀 있으므로 전부 보류하면 도움이 되지 않습니다. 틀리지 않으려고 모든 질문을 보류하는 구현도 답해야 할 질문에서 점수가 낮아져야 합니다.

In [ ]:
if live:
    for label, report in holdout.items():
        row = next(r for r in report["rows"] if r["id"] == "H01")
        print(label, row["run_id"])
        if row["error"]:
            print("실행 오류", row["error"])
        else:
            print(row["output"]["answer"], row["output"]["citations"])

답을 저장한 뒤 `answers/recorder-holdout.csv`와 원문을 엽니다. 각 구현별로 답한 수, 올바르게 보류한 수, 잘못 답한 수, 호출 오류 수를 적습니다. 예를 들어 네 건 중 한 건에서 새로 근거 없는 기능 단정이 생겼다면 나머지 세 건의 문장이 자연스러워졌어도 채택 기준을 위반합니다. 반대로 품질 차이가 관찰되지 않고 실행 수도 적다면 비용이 조금 낮은 것만으로 모든 사용 조건에서 우수하다고 일반화할 수 없습니다.

## 실습 · 똑같이 3개 맞힌 두 후보 중 하나를 선택할 수 있을까

아래는 결정을 연습하기 위해 만든 **가상의 결과**입니다. 네 입력의 답할 수 있는 범위를 먼저 읽고 사례별 피해를 표시하세요. P는 H01에서 불가능한 즉시 변경을 지시했고 나머지 세 건은 맞았습니다. Q는 H01·H02·H03을 맞혔지만 H04에서 설명서에 없는 자동 소음 감소를 지원한다고 단정했습니다. 모두 3/4입니다.

우리 팀의 사전 기준이 ‘근거 없는 기능 지원 단정을 새로 만들면 보류’인 경우와, ‘사용 중 불가능한 조작을 지시하면 보류’인 경우에 결론을 각각 적으세요. 어느 쪽도 충분히 쓸 만하지 않다면 기준선 유지 또는 추가 검사를 선택할 수 있습니다. 이 예시는 바로 앞 셀에서 실제 얻은 점수와 구분해 둡니다.

In [ ]:
# 점수만 같아도 무엇이 달라지는지 두 실패를 사용자 행동 기준으로 적습니다.
review_cases = {}
# P와 Q를 키로 넣고 각 답을 그대로 따른 사용자가 할 행동을 값으로 쓰세요.
# 똑같은 정답 수 아래에 어떤 실패가 각각 가려졌는지 두 값을 읽으세요.

<details class="notebook-answer"><summary>예시 풀이 열기</summary><pre><code>review_cases = {&quot;P&quot;: &quot;녹음 중 바꿀 수 없는 조작을 지시&quot;, &quot;Q&quot;: &quot;원문에 없는 기능을 지원한다고 단정&quot;}
print(review_cases)</code></pre><p>P는 현재 녹음을 계속하면서 표시등 메뉴를 찾게 만듭니다. Q는 근거 없는 기능을 전제로 장비를 고르게 할 수 있습니다. 기능 단정을 금지한 팀은 Q를 보류하고, 잘못된 즉시 조작을 금지한 팀은 P를 보류할 수 있습니다. 같은 기준으로 두 후보를 모두 보류할 수도 있습니다. 실제 결정에서는 기준선도 해당 질문에서 어떻게 답했는지 확인한 뒤 “새로 생긴 실패”인지 적습니다.</p></details>

## 실패한 입력을 회귀 검사로 등록하고 다시 실행하기

**회귀(regression)** 는 전에 되던 행동이 변경 뒤에 깨진 현상입니다. 팀원이 새로 발견한 실패도, 고친 뒤부터는 같은 문제가 되돌아오지 않는지 검사할 수 있습니다. 먼저 팀원의 원래 표현을 복사합니다. ‘분할’을 ‘저장’으로 고쳐 모델이 알아보기 쉽게 해 주면 실제 문제를 감출 수 있습니다.

아래 R01은 진행 방법을 보여 주는 새 질문 예시입니다. A형 1.1판 2쪽에는 저장 공간이 꽉 차면 녹음은 멈추고, 이미 저장한 파일은 남는다고 적혀 있습니다. 자동 덮어쓰기를 단정하면 잘못입니다. 기대 결과를 원문으로 먼저 확인한 후 같은 입력으로 **이전 버전에도** 실행하세요. 둘 다 맞으면 회귀 검사용 새 사례는 되지만 버전 간 회귀가 관찰됐다고 쓰지는 않습니다.

회귀 사례는 다음 변경 때 다시 물을 질문과 기대 결과를 한 쌍으로 보존합니다. `new_case["inputs"]`만 챗봇에 전달하고, 정답 쪽·기대 행동을 담은 `outputs`는 평가할 때 읽습니다. `[new_case]`로 감싸는 이유는 `compare`가 질문 여러 건을 담는 목록을 받기 때문입니다. 사례가 한 건일 때도 각 후보의 답과 실행 ID를 짝지어 비교하세요.

In [ ]:
# 직접 받은 질문으로 바꿀 때는 제품 선택값과 원문에서 확인한 답도 함께 바꿉니다.
new_case = {"inputs": {"id": "R01", "question": "2027년 5월 A형, 꽉 차면 제 앞 녹음부터 덮어써요?",
                      "model": "A", "edition": "1.1"},
            "outputs": {"action": "answer", "source": "recorder-a-1.1.pdf", "page": 2,
                        "reference_answer": "저장 공간이 가득 차면 녹음이 멈추며 이미 저장한 파일은 남는다."}}
if live:
    regression = compare(client, "regression", [new_case], selected, index)
    (work / f"regression-{stamp}.json").write_text(json.dumps(regression, ensure_ascii=False, indent=2), encoding="utf-8")
    for label, report in regression.items():
        row = report["rows"][0]
        print(label, row["run_id"], row["error"] or row["output"]["answer"])

이 세트의 이름과 데이터셋 ID가 새 시험의 위치입니다. 이전 답·새 답 중 어느 쪽이 어떤 조건을 어겼는지 원문으로 판정합니다. 팀의 새 사례를 추가하려면 입력 ID를 R02로 새로 주고 `[new_case, peer_case]`처럼 이전 사례도 포함해 버전을 만드세요. 준비 함수가 행 내용에 따라 새 버전 이름을 부여하므로 예전 실험을 잃지 않습니다.

## 개인 실습 · 동료가 다시 실행할 수 있는 채택 기록

팀원 한 명은 도입 담당, 다른 한 명은 상담 담당입니다. 도입 담당은 기준선·변경 후보의 실험 ID와 실제 설정을 제시합니다. 상담 담당은 H01 또는 H03의 원문 쪽을 읽고 답의 행동 조건을 판정합니다. 두 사람이 독립적으로 채택·보류·추가 확인 중 하나를 고른 뒤 같거나 달랐던 근거를 비교하세요.

결과물은 `my-recorder-evaluation` 아래의 사전 계획, 최종 확인 결과, 회귀 시험 결과와 활동지입니다. 마지막에는 팀원이 같은 노트북을 열어 R01 또는 R02를 기준선에 다시 넣고, 기록된 결과를 재현할 수 있는지 확인합니다. 개인 API 키는 결과물에 포함하지 않습니다.

In [ ]:
# 본인 실험에서 관찰한 사실과 결정, 다시 검토할 조건을 적습니다.
decision = {"결정": "", "실험 ID": "", "근거 질문 ID": "", "원문 근거": "", "다시 점검할 조건": ""}

<details class="notebook-answer"><summary>모범 예시와 판정 근거 보기</summary><p>판단 예시입니다. 후보가 H01에 ‘지금 숨김을 선택하라’고 하고 기준선이 녹음 종료 후 설정을 안내했다면 변경을 보류합니다. B형 1.0판 1쪽은 녹음 중 설정 변경을 허용하지 않습니다. 어떤 버전도 실패하지 않았다면 그 차이를 꾸며내지 말고 실제 차이가 없었던 질문과 다음 재검사 조건을 적습니다.</p></details>

## 팀원이 재현할 질문과 결정 저장하기

중간 프로젝트 팀원에게 개발 과정에서 확인한 질문 한 건과 이를 검토할 쪽을 넘깁니다. 실험 ID와 원문을 직접 확인한 뒤 위 결정표를 채우세요. 입력하지 않으면 검토 대기로 저장합니다.

In [ ]:
case = previous["case"]
verdict = decision["결정"].strip()
known_experiments = {item["experiment_id"] for item in holdout.values()} if live else set()
reviewed = (verdict in {"채택", "보류"} and decision["실험 ID"] in known_experiments
            and all(str(v).strip() for v in decision.values()))
project_case = {**case["inputs"], "reference": case["outputs"], "candidate_k": candidate_k,
                "decision": verdict if reviewed else "검토 대기",
                "evidence_mode": "실험" if reviewed else "기록 읽기",
                "experiment_id": decision["실험 ID"] if reviewed else "",
                "evaluation_note": decision if reviewed else {}}
out = work / "project-handoff.json"
out.write_text(json.dumps(project_case, ensure_ascii=False, indent=2), encoding="utf-8")
print(out.relative_to(root), project_case["decision"])

my-recorder-evaluation/project-handoff.json 검토 대기


**중간 프로젝트 인계**. 팀은 PDF 세 개, 개발·확인·회귀 세트 버전, 실행 노트북, 사용 모델·색인·코드 식별값, 실제 trace와 실험 ID를 한곳에 보관합니다. 다음 담당자는 어떤 실패가 다시 나타나면 배포를 보류해야 하는지 같은 판정 기준으로 확인할 수 있습니다.

[실험 결과 비교하기](https://docs.langchain.com/langsmith/compare-experiment-results) · [실험에서 대상과 평가자 구분하기](https://docs.langchain.com/langsmith/define-target-function)